# Perturbation Prediction Benchmark — Tutorial

This notebook walks you through evaluating perturbation prediction models
on your own `.h5ad` Perturb-seq dataset.

## Supported Models

| Model | Approach | Reference |
|-------|----------|-----------|
| **GEARS** | Graph Auto-Regressive | Roohani et al. 2023 |
| **scGPT** | Transformer (pre-trained) | Cui et al. 2024 |
| **STATE** | Spatial Transcriptomics adapted | Arc Institute |
| **Cell2Sentence** | LLM-based (Gemma-2-2B) | Van Dijk Lab |
| **CPA** | Conditional Perturbation Autoencoder | Lotfollahi et al. 2023 |

## Evaluation Tiers

- **Tier 1**: Whole-profile identity (CA, PDS, Systema Pearson Delta)
- **Tier 2**: DE gene accuracy (Directional Accuracy, Pearson Delta Top-K, Jaccard Top-K)
- **Tier 3**: Distributional fidelity (Energy Distance, MMD)

## Step 1: Setup

Clone the repository and install base dependencies.

In [ ]:
# Clone the repository (skip if already cloned)
# !git clone https://github.com/kagtgi/PerturbationBenchmarkTool.git
# %cd PerturbationBenchmarkTool

# Install base dependencies
!pip install -q anndata scanpy scipy pandas matplotlib torch

## Step 2: Configure

Set the path to your `.h5ad` dataset and choose device/seed.

In [ ]:
from eval import config

# === EDIT THESE ===
config.DATA_PATH = "K562.h5ad"       # Path to your .h5ad Perturb-seq file
config.DEVICE = "cuda"               # "cuda" or "cpu"
config.RANDOM_SEED = 42              # For reproducibility
config.CTRL_LABEL = "non-targeting"  # Control cell label in obs['gene']
config.PERT_COL = "gene"             # Column in obs with perturbation labels
config.OUTPUT_DIR = "results"        # Where to save outputs

print(f"Data:   {config.DATA_PATH}")
print(f"Device: {config.DEVICE}")
print(f"Seed:   {config.RANDOM_SEED}")

## Step 3: Load & Subsample Dataset

The framework loads the `.h5ad` file and performs stratified subsampling
to ensure fair comparison across models. Every model sees the exact same cells.

In [ ]:
from eval.dataset import load_h5ad, ensure_raw_counts
from eval.sampling import stratified_subsample

# Load dataset
adata = load_h5ad(config.DATA_PATH)
adata = ensure_raw_counts(adata)
print(f"Full dataset: {adata.shape}")

# Stratified 20% subsample (identical across all models)
adata_sub = stratified_subsample(adata)
print(f"Subsampled:   {adata_sub.shape}")
print(f"Perturbations: {adata_sub.obs[config.PERT_COL].nunique() - 1}")

## Step 4: Run Individual Model Evaluations

Each model runs in isolation. You can evaluate one at a time
(recommended: **restart runtime** between heavy models to free GPU memory).

### Option A: Run a single model

In [ ]:
from eval.models import run_model_eval
import json, os

# Choose one model at a time: "gears", "scgpt", "state", "cell2sentence", "cpa"
MODEL_NAME = "gears"

cfg = {
    "DATA_PATH": config.DATA_PATH,
    "RANDOM_SEED": config.RANDOM_SEED,
    "DEVICE": config.DEVICE,
    "CTRL_LABEL": config.CTRL_LABEL,
    "PERT_COL": config.PERT_COL,
    "TOP_K_DE": config.TOP_K_DE,
    "MIN_CELLS_PER_PERT": config.MIN_CELLS_PER_PERT,
    "MAX_T3_CELLS": config.MAX_T3_CELLS,
    "OUTPUT_DIR": config.OUTPUT_DIR,
}

result = run_model_eval(MODEL_NAME, adata_sub.copy(), cfg)

# Display results
print(f"\nModel: {result['model']}")
print(f"Perturbations evaluated: {len(result['pert_names'])}")
print(f"Runtime: {result['runtime_seconds']:.1f}s")
print("\nMetrics:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v:.4f}")

# Save result
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
with open(f"{config.OUTPUT_DIR}/{MODEL_NAME}_results.json", "w") as f:
    json.dump(result, f, indent=2, default=str)
print(f"\nSaved to {config.OUTPUT_DIR}/{MODEL_NAME}_results.json")

### Option B: Run all models via eval_runner (recommended)

Each model runs in an **isolated subprocess** — no library conflicts, no runtime restart needed.
Logs stream live to this cell as each model progresses.
If a model fails, the error and traceback are shown automatically at the end.

In [ ]:
from eval.eval_runner import run

# Run all models (or specify a subset)
results = run(
    data_path=config.DATA_PATH,
    models=["gears", "scgpt", "state", "cell2sentence", "cpa"],
    device=config.DEVICE,
    seed=config.RANDOM_SEED,
    output_dir=config.OUTPUT_DIR,
    # isolate=True,  # default: each model in separate subprocess (recommended)
)

# Quick status check
for r in results:
    status = r.get('status', 'unknown')
    n_pert = len(r.get('pert_names', []))
    runtime = r.get('runtime_seconds', 0) / 60
    print(f"  {r['model']:20s}  {status:10s}  perts={n_pert}  {runtime:.1f} min")

## Step 5: Collect & Compare Results

After running all models, aggregate results into a comparison table.

In [ ]:
from eval.collect_results import collect, pretty_table

df = collect(config.OUTPUT_DIR)
print(pretty_table(df))

## Step 6: Interpret Results

The output table shows 8 metrics across 3 tiers:

### Tier 1 — Whole-Profile Identity
- **Centroid Accuracy (CA)** ↑: Does the predicted centroid match the correct perturbation?
- **Profile Distance Score (PDS)** ↓: Relative distance to the true centroid vs others.
- **Systema Pearson Delta** ↑: Genome-wide correlation of expression changes.

### Tier 2 — DE Gene Accuracy
- **Directional Accuracy** ↑: Are up/down-regulated genes predicted correctly?
- **Pearson Delta Top-K** ↑: Correlation of top-K DE gene changes.
- **Jaccard Top-K** ↑: Overlap of predicted vs true top-K DE gene sets.

### Tier 3 — Distributional Fidelity
- **Energy Distance** ↓: Statistical distance between predicted and true cell distributions.
- **MMD (RBF)** ↓: Kernel-based distributional divergence.

**↑ = higher is better, ↓ = lower is better**

In [ ]:
# View as DataFrame for further analysis
df

In [ ]:
# Export to CSV
df.to_csv(f"{config.OUTPUT_DIR}/final_comparison.csv", index=False)
print(f"Saved to {config.OUTPUT_DIR}/final_comparison.csv")

---
## Troubleshooting

If any model fails, use the built-in debug helpers to diagnose the issue.


In [ ]:
# ── Debug helpers ────────────────────────────────────────────────────────
from eval.eval_runner import print_errors, print_logs

# 1. Show error message + traceback for every failed model:
print_errors(config.OUTPUT_DIR)

# 2. Show full subprocess log (stdout + stderr) for a specific model:
# print_logs(config.OUTPUT_DIR, model='gears')

# 3. Show logs for ALL models at once:
# print_logs(config.OUTPUT_DIR)

# 4. Inspect raw result JSON:
# import json
# print(json.dumps(json.load(open(f'{config.OUTPUT_DIR}/gears_results.json')), indent=2))


### Common Issues

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| All 5 models fail in < 5 s | Not running from project root | `cd PerturbationBenchmarkTool` |
| `ModuleNotFoundError: No module named 'eval'` | Wrong working directory | See above |
| `CUDA not available` | Cell2Sentence needs GPU | Use a GPU runtime or skip `cell2sentence` |
| Gene overlap = 0 (CPA) | Control label mismatch | Set `config.CTRL_LABEL` to match your data |
| Timeout | Model too slow | Increase timeout or reduce `SUBSAMPLE_FRAC` in config |
